# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score

## 1. Preprocessing

In [2]:
df_ns = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')
df_ns['dayofweek'] = df_scaled['dayofweek'].values

X = df_ns.drop('dayofweek', axis=1).values
y = df_ns['dayofweek'].values

In [3]:
# First split: train+valid vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

# Second split: train vs valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=21, stratify=y_trainval
)

print('Train:', X_train.shape)
print('Valid:', X_valid.shape)
print('Test:', X_test.shape)

Train: (1078, 43)
Valid: (270, 43)
Test: (338, 43)


In [4]:
def print_metrics(model, X_val, y_val):
    y_pred = model.predict(X_val)
    print(f'accuracy is {accuracy_score(y_val, y_pred):.5f}')
    print(f'precision is {precision_score(y_val, y_pred, average="weighted"):.5f}')
    print(f'recall is {recall_score(y_val, y_pred, average="weighted"):.5f}')

## 2. Individual classifiers

In [5]:
# SVM
svm = SVC(kernel='rbf', C=10, gamma='auto', probability=True, random_state=21)
svm.fit(X_train, y_train)
print('SVM:')
print_metrics(svm, X_valid, y_valid)

SVM:
accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [6]:
# Decision Tree
tree = DecisionTreeClassifier(max_depth=21, class_weight='balanced', criterion='gini', random_state=21)
tree.fit(X_train, y_train)
print('Decision Tree:')
print_metrics(tree, X_valid, y_valid)

Decision Tree:
accuracy is 0.86667
precision is 0.87170
recall is 0.86667


In [7]:
# Random Forest
rf = RandomForestClassifier(n_estimators=50, max_depth=22, criterion='entropy', random_state=21)
rf.fit(X_train, y_train)
print('Random Forest:')
print_metrics(rf, X_valid, y_valid)

Random Forest:
accuracy is 0.90741
precision is 0.90781
recall is 0.90741


## 3. Voting classifiers

In [8]:
# Try different voting weights
best_voting_acc = 0
best_voting_prec = 0
best_weights = None

for weights in [(1,1,1), (1,1,2), (1,2,1), (2,1,1), (1,1,3), (1,2,3)]:
    voting = VotingClassifier(
        estimators=[('svm', svm), ('tree', tree), ('rf', rf)],
        voting='soft', weights=list(weights)
    )
    voting.fit(X_train, y_train)
    y_pred = voting.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    print(f'weights={weights}: acc={acc:.5f}, prec={prec:.5f}')
    if acc > best_voting_acc or (acc == best_voting_acc and prec > best_voting_prec):
        best_voting_acc = acc
        best_voting_prec = prec
        best_weights = weights

print(f'\nBest weights: {best_weights}')

weights=(1, 1, 1): acc=0.88519, prec=0.88882
weights=(1, 1, 2): acc=0.90370, prec=0.90551
weights=(1, 2, 1): acc=0.86667, prec=0.87170
weights=(2, 1, 1): acc=0.90370, prec=0.90564
weights=(1, 1, 3): acc=0.90000, prec=0.90105
weights=(1, 2, 3): acc=0.87778, prec=0.88161

Best weights: (2, 1, 1)


In [9]:
# Best voting on test set
best_voting = VotingClassifier(
    estimators=[('svm', svm), ('tree', tree), ('rf', rf)],
    voting='soft', weights=list(best_weights)
)
best_voting.fit(X_train, y_train)
print('Best Voting on TEST:')
print_metrics(best_voting, X_test, y_test)

Best Voting on TEST:
accuracy is 0.89645
precision is 0.89966
recall is 0.89645


## 4. Bagging classifiers

In [10]:
best_bag_acc = 0
best_bag_prec = 0
best_bag_n = None

for n_est in [5, 10, 20, 50]:
    bagging = BaggingClassifier(
        estimator=SVC(kernel='rbf', C=10, gamma='auto', probability=True, random_state=21),
        n_estimators=n_est, random_state=21
    )
    bagging.fit(X_train, y_train)
    y_pred = bagging.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    print(f'n_estimators={n_est}: acc={acc:.5f}, prec={prec:.5f}')
    if acc > best_bag_acc or (acc == best_bag_acc and prec > best_bag_prec):
        best_bag_acc = acc
        best_bag_prec = prec
        best_bag_n = n_est

print(f'\nBest n_estimators: {best_bag_n}')

n_estimators=5: acc=0.87407, prec=0.88423
n_estimators=10: acc=0.88519, prec=0.89427
n_estimators=20: acc=0.88519, prec=0.89258
n_estimators=50: acc=0.88148, prec=0.89035

Best n_estimators: 10


In [11]:
# Best bagging on test set
best_bagging = BaggingClassifier(
    estimator=SVC(kernel='rbf', C=10, gamma='auto', probability=True, random_state=21),
    n_estimators=best_bag_n, random_state=21
)
best_bagging.fit(X_train, y_train)
print('Best Bagging on TEST:')
print_metrics(best_bagging, X_test, y_test)

Best Bagging on TEST:
accuracy is 0.86391
precision is 0.86966
recall is 0.86391


## 5. Stacking classifiers

In [12]:
best_stack_acc = 0
best_stack_prec = 0
best_stack_params = None

for n_splits in [2, 3, 4, 5, 6, 7]:
    for passthrough in [True, False]:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        stacking = StackingClassifier(
            estimators=[('svm', svm), ('tree', tree), ('rf', rf)],
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv, passthrough=passthrough
        )
        stacking.fit(X_train, y_train)
        y_pred = stacking.predict(X_valid)
        acc = accuracy_score(y_valid, y_pred)
        prec = precision_score(y_valid, y_pred, average='weighted')
        print(f'n_splits={n_splits}, passthrough={passthrough}: acc={acc:.5f}, prec={prec:.5f}')
        if acc > best_stack_acc or (acc == best_stack_acc and prec > best_stack_prec):
            best_stack_acc = acc
            best_stack_prec = prec
            best_stack_params = (n_splits, passthrough)

print(f'\nBest params: n_splits={best_stack_params[0]}, passthrough={best_stack_params[1]}')

n_splits=2, passthrough=True: acc=0.90000, prec=0.90100
n_splits=2, passthrough=False: acc=0.90370, prec=0.90438
n_splits=3, passthrough=True: acc=0.90370, prec=0.90660
n_splits=3, passthrough=False: acc=0.90741, prec=0.90996
n_splits=4, passthrough=True: acc=0.91111, prec=0.91348
n_splits=4, passthrough=False: acc=0.90741, prec=0.91098
n_splits=5, passthrough=True: acc=0.90741, prec=0.90975
n_splits=5, passthrough=False: acc=0.91481, prec=0.91710
n_splits=6, passthrough=True: acc=0.90741, prec=0.90962
n_splits=6, passthrough=False: acc=0.90741, prec=0.90994
n_splits=7, passthrough=True: acc=0.90370, prec=0.90666
n_splits=7, passthrough=False: acc=0.90370, prec=0.90480

Best params: n_splits=5, passthrough=False


In [13]:
# Best stacking on test set
cv = StratifiedKFold(n_splits=best_stack_params[0], shuffle=True, random_state=21)
best_stacking = StackingClassifier(
    estimators=[('svm', svm), ('tree', tree), ('rf', rf)],
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=cv, passthrough=best_stack_params[1]
)
best_stacking.fit(X_train, y_train)
print('Best Stacking on TEST:')
print_metrics(best_stacking, X_test, y_test)

Best Stacking on TEST:
accuracy is 0.90828
precision is 0.91087
recall is 0.90828


## 6. Predictions

In [14]:
# Choose best overall model
models_eval = {
    'Voting': best_voting,
    'Bagging': best_bagging,
    'Stacking': best_stacking
}

best_name = None
best_acc = 0
best_prec = 0
for name, model in models_eval.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    print(f'{name}: acc={acc:.5f}, prec={prec:.5f}')
    if acc > best_acc or (acc == best_acc and prec > best_prec):
        best_acc = acc
        best_prec = prec
        best_name = name

print(f'\nBest model: {best_name}')
best_ensemble = models_eval[best_name]

Voting: acc=0.89645, prec=0.89966
Bagging: acc=0.86391, prec=0.86966
Stacking: acc=0.90828, prec=0.91087

Best model: Stacking


In [15]:
# Error analysis
y_pred = best_ensemble.predict(X_test)
errors = pd.DataFrame({'true': y_test, 'pred': y_pred})
errors['wrong'] = errors['true'] != errors['pred']
print('Error rate per weekday:')
print(errors.groupby('true')['wrong'].mean().sort_values(ascending=False))

submits = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])
submits['dayofweek'] = submits['timestamp'].dt.dayofweek
full_pred = best_ensemble.predict(X)
submits['wrong'] = full_pred != submits['dayofweek'].values

print('\nError rate per labname:')
print(submits.groupby('labname')['wrong'].mean().sort_values(ascending=False).head())

print('\nError rate per uid:')
print(submits.groupby('uid')['wrong'].mean().sort_values(ascending=False).head())

Error rate per weekday:
true
0    0.296296
5    0.129630
4    0.095238
6    0.084507
1    0.072727
2    0.066667
3    0.025000
Name: wrong, dtype: float64

Error rate per labname:
labname
lab03      1.000000
laba06s    0.098361
laba04s    0.086538
laba06     0.062500
laba04     0.056180
Name: wrong, dtype: float64

Error rate per uid:
uid
user_22    0.428571
user_23    0.250000
user_6     0.250000
user_17    0.088235
user_30    0.076923
Name: wrong, dtype: float64


In [16]:
# Save model
with open('../data/ensembles_model.pkl', 'wb') as f:
    pickle.dump(best_ensemble, f)
print('Model saved!')

Model saved!
